In [1]:
import pandas as pd
import warnings

# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def backtest_trades(price_data, signal_data, tp_buy=None, tp_sell=None, sl_buy=None, sl_sell=None, entry_time_offset=None, percentage_change=None, time_limit_minutes=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Total NAV'
    ])
    
    initial_nav = 100000  # Starting initial NAV

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']

        if signal_value == 0:
            continue
        
        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']
        
        half_initial_nav = initial_nav / 2
        total_nav = 0  # Initialize total NAV for this signal
        pnl_sum = 0    # Sum of pnl for ROI calculation

        for side in ['Buy', 'Sell']:
            entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset)
            if entry_datetime is None:
                new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Signal Open Price': signal_open_price,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Not Filled',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00',
                    'ROI': 0,
                    'NAV': initial_nav,  # No change in NAV
                    'Total NAV': initial_nav  # No change in total NAV
                }])
                output_data = pd.concat([output_data, new_row], ignore_index=True)
                continue

            # Step 1: initial margin * (1 - 0.0002) = M1
            M1 = half_initial_nav * (1 - 0.0002)
            
            if side == 'Buy':
                tp_price = entry_price * (1 + tp_buy)
                sl_price = entry_price * (1 - sl_buy)
            else:
                tp_price = entry_price * (1 - tp_sell)
                sl_price = entry_price * (1 + sl_sell)
            
            result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)

            # Step 2: based on TP or SL hit, calculate M2
            if result == 1:
                M2 = M1 * (1 + (tp_buy if side == 'Buy' else tp_sell))
            elif result == -1:
                M2 = M1 * (1 - (sl_buy if side == 'Buy' else sl_sell))
            else:
                M2 = M1

            # Step 3: M3 = M2 * (1 - 0.0005)
            M3 = M2 * (1 - 0.0005)
            
            # Step 4: Calculate PnL
            pnl = M3 - half_initial_nav
            pnl_sum += pnl
            
            # Update total NAV for this signal
            total_nav += M3
            
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': entry_price,
                'TP Price': tp_price,
                'SL Price': sl_price,
                'Result': 'TP Hit' if result == 1 else 'SL Hit',
                'Duration': duration_str,
                'Execution Latency': format_duration(entry_duration),
                'ROI': 0,  # ROI will be calculated based on total NAV
                'NAV': M3,
                'Total NAV': 0  # Placeholder for total NAV
            }])
            
            
            output_data = pd.concat([output_data, new_row], ignore_index=True)
        
        # Step 5: Calculate final NAV of the two positions
        final_nav = initial_nav + pnl_sum
        
        # Update the total NAV for the signal in all relevant rows
        output_data.loc[output_data['Datetime'] == signal_datetime, 'Total NAV'] = final_nav
        
        # Step 6: Calculate ROI based on final NAV
        roi = ((final_nav - initial_nav) / initial_nav) * 100
        output_data.loc[output_data['Datetime'] == signal_datetime, 'ROI'] = roi
        
        # Update the initial NAV for the next signal
        initial_nav = final_nav

    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['Total NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000
    
    for date, nav in daily_nav.items():
        daily_return = (nav - previous_day_nav) / previous_day_nav * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)    
    
    return output_data


# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None


In [2]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\Signature_AI_Results_Final.csv', parse_dates=['Datetime'])

In [ ]:
# # Define the parameters
# tp_buy = 0.005
# sl_buy = 0.01
# tp_sell = 0.005
# sl_sell = 0.01
# entry_time_offset = 0 # Time offset in minutes
# percentage_change = 0
# time_limit_minutes = 120 
# # Call the backtest_trades function
# backtest_output = backtest_trades(
#     price_data=price_data,
#     signal_data=signal_data,
#     tp_buy = 0.005,
#     sl_buy = 0.01,
#     tp_sell = 0.005,
#     sl_sell = 0.01,
#     entry_time_offset=entry_time_offset,
#     percentage_change=percentage_change,
#     time_limit_minutes=time_limit_minutes
# )
# 
# backtest_output

In [3]:
import numpy as np
import random

np.random.seed(42)
random.seed(42)

# Define the parameter ranges
tp_values = np.arange(0.003, 0.007, 0.001)

# Function to generate sl_values based on tp
def generate_sl_values(tp):
    sl_values = np.arange(tp + 0.001, tp * 2, 0.001)  # Adjust upper bound to ensure non-empty range
    return sl_values

# Evaluation function
def evaluate(individual, interval_price_data, interval_signal_data):
    tp, sl = individual
    result = backtest_trades(
        interval_price_data, interval_signal_data, tp_buy=tp, tp_sell=tp, sl_buy=sl, sl_sell=sl,
        entry_time_offset=60, percentage_change=0, time_limit_minutes=120
    )

    final_nav = result['Total NAV'].iloc[-1]
    roi = ((final_nav - 100000) / 100000) * 100

    return roi

# Randomly initialize an individual
def create_individual():
    tp = np.random.choice(tp_values)
    sl_values = generate_sl_values(tp)
    sl = np.random.choice(sl_values)
    return [tp, sl]


def mutate(individual):
    index = random.randint(0, len(individual) - 1)
    if index == 0:
        tp = np.random.choice(tp_values)
        sl_values = generate_sl_values(tp)
        individual[0] = tp
        individual[1] = np.random.choice(sl_values)
    elif index == 1:
        tp = individual[0]
        sl_values = generate_sl_values(tp)
        individual[1] = np.random.choice(sl_values)
    return individual

# Simulated Annealing algorithm
def simulated_annealing(interval_price_data, interval_signal_data):
    current_individual = create_individual()
    current_fitness = evaluate(current_individual, interval_price_data, interval_signal_data)
    best_individual = list(current_individual)
    best_fitness = current_fitness

    initial_temperature = 1.0
    final_temperature = 0.001
    alpha = 0.99
    temperature = initial_temperature

    while temperature > final_temperature:
        new_individual = mutate(list(current_individual))
        new_fitness = evaluate(new_individual, interval_price_data, interval_signal_data)

        if new_fitness > current_fitness or random.uniform(0, 1) < np.exp((new_fitness - current_fitness) / temperature):
            current_individual = new_individual
            current_fitness = new_fitness

        if current_fitness > best_fitness:
            best_individual = list(current_individual)
            best_fitness = current_fitness

        temperature *= alpha

    best_tp, best_sl = best_individual
    optimized_roi = best_fitness

    print(f"Optimized parameters:")
    print(f"Best Take Profit: {best_tp}")
    print(f"Best Stop Loss: {best_sl}")
    print(f"Optimized ROI: {optimized_roi:.4f}")

    return best_individual

In [4]:
def optimize_and_backtest_intervals(interval_type=None):
    if interval_type == 'weekly':
        intervals = pd.date_range(start='2024-01-01', end='2024-07-01', freq='W')
    elif interval_type == 'monthly':
        intervals = pd.date_range(start='2024-01-01', end='2024-07-01', freq='MS')

    reports = []

    for i in range(1, len(intervals) - 1):
        start_date = intervals[i - 1]
        end_date = intervals[i]
        interval_price_data = price_data[start_date:end_date]
        interval_signal_data = signal_data[
            (signal_data['Datetime'] >= start_date) & (signal_data['Datetime'] < end_date)]
        
        # Debug: Print data shapes
        print(f"Interval {i}: start_date={start_date}, end_date={end_date}")
 
        
        best_individual = simulated_annealing(interval_price_data, interval_signal_data)

        print(i, best_individual)
        tp, sl = best_individual

        next_start_date = intervals[i]
        next_end_date = intervals[i + 1]

        next_interval_price_data = price_data[next_start_date:next_end_date]
        next_interval_signal_data = signal_data[
            (signal_data['Datetime'] >= next_start_date) & (signal_data['Datetime'] < next_end_date)]
        
        # Debug: Print data shapes
        print(f"Next Interval {i}: next_start_date={next_start_date}, next_end_date={next_end_date}")
   

        result = backtest_trades(
            next_interval_price_data, next_interval_signal_data,  tp_buy=tp, tp_sell=tp, sl_buy=sl, sl_sell=sl,
            entry_time_offset=60,
            percentage_change=0, time_limit_minutes=120
        )

        total_trades = len(result[result['Result'].isin(['TP Hit', 'SL Hit'])])
        wins = len(result[result['Result'] == 'TP Hit'])
        losses = len(result[result['Result'] == 'SL Hit'])
        win_percentage = wins / total_trades if total_trades > 0 else 0
        final_nav = result['Total NAV'].iloc[-1]
        daily_nav = result.set_index('Datetime')['Daily Return']

        for nav_date, daily_nav in daily_nav.items():
            report = {
                'Date': nav_date,
                'Start Date': next_start_date,
                'End Date': next_end_date,
                'Total Trades': total_trades,
                'Wins': wins,
                'Losses': losses,
                'WinRate': win_percentage,
                'Final NAV': final_nav,
                'Daily Return': daily_nav,
                'TP': tp,
                'SL': sl
            }
            reports.append(report)

    report_df = pd.DataFrame(reports)
    report_df['Date'] = pd.to_datetime(report_df['Date'])
    report_df.set_index('Date', inplace=True)
    return report_df

In [5]:
# Run the function again to verify the intervals
report_df = optimize_and_backtest_intervals(interval_type='monthly')
report_df.to_csv('E:\Signal Backtesting\Output\Monthly_optimization_with_tow_positions_immediate.csv')

Interval 1: start_date=2024-01-01 00:00:00, end_date=2024-02-01 00:00:00
Optimized parameters:
Best Take Profit: 0.006
Best Stop Loss: 0.01
Optimized ROI: 2.5380
1 [0.006, 0.01]
Next Interval 1: next_start_date=2024-02-01 00:00:00, next_end_date=2024-03-01 00:00:00
Interval 2: start_date=2024-02-01 00:00:00, end_date=2024-03-01 00:00:00
Optimized parameters:
Best Take Profit: 0.005
Best Stop Loss: 0.007
Optimized ROI: -0.0504
2 [0.005, 0.007]
Next Interval 2: next_start_date=2024-03-01 00:00:00, next_end_date=2024-04-01 00:00:00
Interval 3: start_date=2024-03-01 00:00:00, end_date=2024-04-01 00:00:00
Optimized parameters:
Best Take Profit: 0.006
Best Stop Loss: 0.011
Optimized ROI: -3.0172
3 [0.006, 0.011]
Next Interval 3: next_start_date=2024-04-01 00:00:00, next_end_date=2024-05-01 00:00:00
Interval 4: start_date=2024-04-01 00:00:00, end_date=2024-05-01 00:00:00
Optimized parameters:
Best Take Profit: 0.006
Best Stop Loss: 0.011
Optimized ROI: 0.8795
4 [0.006, 0.011]
Next Interval 4:

In [ ]:
import pandas as pd


def backtest_trades(price_data, signal_data, tp_buy=None, tp_sell=None, sl_buy=None, sl_sell=None,
                    entry_time_offset=None, percentage_change=None, time_limit_minutes=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Total NAV'
    ])

    initial_nav = 100000  # Starting initial NAV

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']

        if signal_value == 0:
            continue

        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']

        half_initial_nav = initial_nav / 2
        total_nav = 0  # Initialize total NAV for this signal
        pnl_sum = 0  # Sum of pnl for ROI calculation

        for side in ['Buy', 'Sell']:
            entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime,
                                                                          percentage_change, side, time_limit_minutes,
                                                                          entry_time_offset)
            if entry_datetime is None:
                new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Signal Open Price': signal_open_price,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Not Filled',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00',
                    'ROI': 0,
                    'NAV': initial_nav,  # No change in NAV
                    'Total NAV': initial_nav  # No change in total NAV
                }])
                output_data = pd.concat([output_data, new_row], ignore_index=True)
                continue

            # Step 1: initial margin * (1 - 0.0002) = M1
            M1 = half_initial_nav * (1 - 0.0002)

            if side == 'Buy':
                tp_price = entry_price * (1 + tp_buy)
                sl_price = entry_price * (1 - sl_buy)
            else:
                tp_price = entry_price * (1 - tp_sell)
                sl_price = entry_price * (1 + sl_sell)

            result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)

            # Step 2: based on TP or SL hit, calculate M2
            if result == 1:
                M2 = M1 * (1 + (tp_buy if side == 'Buy' else tp_sell))
            elif result == -1:
                M2 = M1 * (1 - (sl_buy if side == 'Buy' else sl_sell))
            else:
                M2 = M1

            # Step 3: M3 = M2 * (1 - 0.0005)
            M3 = M2 * (1 - 0.0005)

            # Step 4: Calculate PnL
            pnl = M3 - half_initial_nav
            pnl_sum += pnl

            # Update total NAV for this signal
            total_nav += M3

            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': entry_price,
                'TP Price': tp_price,
                'SL Price': sl_price,
                'Result': 'TP Hit' if result == 1 else 'SL Hit',
                'Duration': duration_str,
                'Execution Latency': format_duration(entry_duration),
                'ROI': 0,  # ROI will be calculated based on total NAV
                'NAV': M3,
                'Total NAV': 0  # Placeholder for total NAV
            }])

            output_data = pd.concat([output_data, new_row], ignore_index=True)

        # Step 5: Calculate final NAV of the two positions
        final_nav = initial_nav + pnl_sum

        # Update the total NAV for the signal in all relevant rows
        output_data.loc[output_data['Datetime'] == signal_datetime, 'Total NAV'] = final_nav

        # Step 6: Calculate ROI based on final NAV
        roi = ((final_nav - initial_nav) / initial_nav) * 100
        output_data.loc[output_data['Datetime'] == signal_datetime, 'ROI'] = roi

        # Update the initial NAV for the next signal
        initial_nav = final_nav

    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['Total NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000

    for date, nav in daily_nav.items():
        daily_return = (nav - previous_day_nav) / previous_day_nav * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)

    return output_data


# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"


def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if (price_row['High'] >= tp_price).any():
                result = 1
                exit_datetime = current_datetime
                break
            elif (price_row['Low'] <= sl_price).any():
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if (price_row['Low'] <= tp_price).any():
                result = 1
                exit_datetime = current_datetime
                break
            elif (price_row['High'] >= sl_price).any():
                result = -1
                exit_datetime = current_datetime
                break

    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'

    return result, duration_str


def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None

    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (
                1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)

    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None


price_data = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\Signature_AI_Results_Final.csv', parse_dates=['Datetime'])
# # Define the parameters
tp_buy = 0.005
sl_buy = 0.009
tp_sell = 0.005
sl_sell = 0.009
entry_time_offset = 60 # Time offset in minutes
percentage_change = 0
time_limit_minutes = 120 
# Call the backtest_trades function
backtest_output = backtest_trades(
    price_data=price_data,
    signal_data=signal_data,
    tp_buy = tp_buy,
    sl_buy = sl_buy,
    tp_sell = tp_sell,
    sl_sell = sl_sell,
    entry_time_offset=entry_time_offset,
    percentage_change=percentage_change,
    time_limit_minutes=time_limit_minutes
)
